### 🎲 Initial Setup

In [0]:
import pyspark.sql.functions as sf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from evidently import Report, Dataset, DataDefinition
from evidently.presets import DataDriftPreset

# style like R ggplot
plt.style.use("ggplot")

In [0]:
# base volume path
BASE_DIR = "/Volumes/workspace/default/home-credit-default-risk"

REPORT_DIR = f"{BASE_DIR}/evidently"
REPORT_PATH = f"{REPORT_DIR}/train_test_drift_report.html"

In [0]:
# reading data from volume
app_train = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true") # since data is not big I can infer schema
    .csv(f"{BASE_DIR}/application_train.csv")
)

app_test = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BASE_DIR}/application_test.csv")
)

In [0]:
# shape
print(f"Application Train shape({app_train.count()}, {len(app_train.columns)})")
print(f"Application Test shape({app_test.count()}, {len(app_test.columns)})")

In [0]:
# target variable
print(f"Target feature: {set(app_train.columns) - set(app_test.columns)}")

### 📬 Applications (Train + Test)
The `application_train` and `application_test` tables form the main starting point of the dataset, where every row represents a single, primary loan request submitted by a client to Home Credit.

This table holds all the personal and financial information collected at the exact moment of application-including the applicant's age, income, employment status, education, family size, and housing type, as well as the specific loan terms requested (like the requested credit amount and monthly annuity). The train dataset includes the target column indicating whether the borrower defaulted, while the test dataset contains the new applications for which predictions are made.

#### Primary Key: SK_ID_CURR
SK_ID_CURR is the Primary Key for the main applicant table (application_train / application_test) and acts as the Foreign Key connecting almost all supplementary tables across the dataset.

---

#### 🤔 Understanding Repayment Risk in Home Credit

##### Measuring Behavior, Not Past Approvals
Repayment risk focuses on what happens *after* the money leaves the bank. Since every applicant in this dataset was already approved, we aren't predicting whether they can get a loan we are predicting the probability that an approved borrower will fail to make their agreed-upon payments once they receive the funds.

##### What `TARGET = 1` Actually Means
In the Home Credit dataset, a borrower gets labeled as `TARGET = 1` if they experienced severe payment difficulties. Specifically, this means they fell significantly late on one or more of their very first monthly installments after taking out the loan.

##### How Financial Institutions Use This Score

* **High-Risk Applicants:** When the model predicts a high risk probability, the lender takes protective measures. They might reject the application outright, approve a smaller loan amount than requested, or require higher interest rates to cover potential losses.
* **Low-Risk Applicants:** When the model predicts a low risk probability, the lender can streamline the process—auto-approving the loan, offering higher credit limits, or providing lower interest rates to win the customer's business.

In [0]:
# target distribution
target_counts_df = (
    app_train
    .groupBy("TARGET")
    .count()
    .orderBy("TARGET")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=target_counts_df,
    x="TARGET",
    y="count",
    hue="TARGET",
    legend=False,
    palette="Reds",
    edgecolor="black",
    ax=ax
)

ax.set_title(
    "Distribution of TARGET",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("TARGET", fontsize=12)
ax.set_ylabel("Total Record Count", fontsize=12)

ax.grid(axis="y", linestyle="--", alpha=0.7)

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="{:,.0f}",
        padding=3
    )

fig.tight_layout()
plt.show()

*Conclusion: The target feature is highly imbalanced.*

---

#### What `NAME_CONTRACT_TYPE` Represents?

##### 1. The Core Meaning
`NAME_CONTRACT_TYPE` specifies the **financial structure** of the loan product the applicant is requesting from Home Credit. It dictates how funds are disbursed, how payments are structured, and how risk behaves over time.

##### 2. Breakdown of Contract Types

* **Cash Loans (Term/Installment Credit):**
  * **How it works:** The borrower receives a one-time lump sum of money directly into their bank account or in cash.
  * **Repayment structure:** Paid back through a fixed number of monthly installments over a set period (e.g., 12, 24, or 36 months).
  * **Risk Profile:** The total balance decreases consistently each month as installments are paid, meaning the bank's maximum exposure shrinks over time.

* **Revolving Loans (Flexible Lines of Credit / Credit Cards):**
  * **How it works:** The borrower is granted an open-ended credit limit that they can draw from, pay down, and reuse repeatedly.
  * **Repayment structure:** Payments vary month-to-month based on active balance and minimum payment requirements; there is no fixed maturity date.
  * **Risk Profile:** Exposure fluctuates based on ongoing borrower spending behavior, making delinquency risk more dynamic compared to fixed installment loans.

In [0]:
contract_type_counts_df = (
    app_train
    .groupBy("NAME_CONTRACT_TYPE")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=contract_type_counts_df,
    x="NAME_CONTRACT_TYPE",
    y="count",
    hue="NAME_CONTRACT_TYPE",
    legend=False,
    palette="pastel",
    edgecolor="black",
    ax=ax,
    hatch="/",
)

ax.set_title(
    "Distribution of NAME_CONTRACT_TYPE",
    fontsize=14,
    fontweight="bold",
    pad=15,
)

ax.set_xlabel("Contract Type", fontsize=12)
ax.set_ylabel("Total Record Count", fontsize=12)

ax.grid(axis="y", linestyle="--", alpha=0.7)

# Add value labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="{:,.0f}",
        padding=3
    )

fig.tight_layout()
plt.show()

---

In [0]:
app_train.select("CODE_GENDER").distinct().show()

#### CODE_GENDER (Applicant Gender)

* **What it is:** The reported legal gender of the applicant submitting the loan request (`M` for Male, `F` for Female, and a handful of `XNA` values for missing/unrecorded entries or maybe for applicants who are not comfortable to reveal their gender).

In [0]:
gender_counts_df = (
    app_train
    .groupBy("CODE_GENDER")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(8, 6))

# pie chart
ax.pie(
    gender_counts_df["count"],
    labels=gender_counts_df["CODE_GENDER"],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"edgecolor": "black"},
)

ax.set_title(
    "Distribution of CODE_GENDER",
    fontsize=14,
    fontweight="bold",
    pad=15,
)

plt.tight_layout()
plt.show()

#### Gender vs. Repayment Risk Analysis

In [0]:
gender_default_df = (
    app_train
    .groupBy("CODE_GENDER")
    .agg(
        sf.count("*").alias("total_applicants"),
        sf.sum("TARGET").alias("total_defaults"),
        sf.round(sf.mean("TARGET") * 100, 2).alias("default_rate_pct")
    )
    .orderBy("default_rate_pct", ascending=False)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=gender_default_df,
    x="CODE_GENDER",
    y="default_rate_pct",
    hue="CODE_GENDER",
    legend=False,
    palette="colorblind",
    edgecolor="black",
    ax=ax,
)

ax.set_title(
    "Default Rate by Gender",
    fontsize=14,
    fontweight="bold",
    pad=15,
)

ax.set_xlabel("Gender", fontsize=12)
ax.set_ylabel("Default Rate (%)", fontsize=12)

ax.grid(axis="y", linestyle="--", alpha=0.7)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f%%",
        padding=3,
    )

fig.tight_layout()
plt.show()

Well, I think due to behavioral economics studies men on average, tend to display higher risk tolerance and financial overconfidence. In consumer credit, this translates to taking on larger debt burdens relative to income or borrowing for discretionary/lifestyle spending rather than necessity.

---

#### FLAG_OWN_CAR (Car Ownership)
What it represents: A binary indicator (Y for Yes, N for No) showing whether the applicant owns an automobile.

Domain Context: Car ownership indicates discretionary income and liquid assets. However, cars also introduce ongoing liabilities (fuel, maintenance, insurance, depreciation).

Data Nuance: In the application dataset, if FLAG_OWN_CAR is Y, you will often find an associated numeric column, OWN_CAR_AGE, indicating how old the applicant's vehicle is (older cars carry higher maintenance overhead and lower collateral value).

In [0]:
app_train.groupBy("FLAG_OWN_CAR").count().show()

In [0]:
car_counts_df = (
    app_train
    .groupBy("FLAG_OWN_CAR")
    .count()
    .orderBy("FLAG_OWN_CAR")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(7, 5))

ax.pie(
    car_counts_df["count"],
    labels=car_counts_df["FLAG_OWN_CAR"],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "width": 0.4,
        "edgecolor": "black"
    },
    colors=[
        "orange", 
        "purple"
    ],
)

ax.set_title(
    "Distribution of Car Ownership",
    fontsize=14,
    fontweight="bold",
    pad=15
)

plt.tight_layout()
plt.show()

#### Gender on Automobile Ownership

In [0]:
# crosstab
cross_df = (
    app_train
    .crosstab("CODE_GENDER", "FLAG_OWN_CAR")
)

cross_pct = (
    cross_df
    .withColumn(
        "total",
        sf.col("N") + sf.col("Y")
    )
    .withColumn(
        "N_pct",
        sf.round(sf.col("N") / sf.col("total") * 100, 2)
    )
    .withColumn(
        "Y_pct",
        sf.round(sf.col("Y") / sf.col("total") * 100, 2)
    )
)

cross_pct.show()

In [0]:
cross_pct_pd = (
    cross_pct
    .select("CODE_GENDER_FLAG_OWN_CAR", "N_pct", "Y_pct")
    .toPandas()
)

heatmap_df = (
    cross_pct_pd
    .set_index("CODE_GENDER_FLAG_OWN_CAR")
    .rename(columns={
        "N_pct": "No Car",
        "Y_pct": "Owns Car"
    })
)

fig, ax = plt.subplots(figsize=(8, 5))

sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".2f",
    cmap="Reds",
    linewidths=0.5,
    linecolor="black",
    cbar_kws={"label": "Percentage (%)"},
    ax=ax
)

ax.set_title(
    "Car Ownership by Gender",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Car Ownership")
ax.set_ylabel("Gender")

plt.tight_layout()
plt.show()

#### Repayment Risk on Owning a Car

In [0]:
# Crosstab
cross_df = app_train.crosstab("FLAG_OWN_CAR", "TARGET")

# Calculate percentages within each car ownership group
cross_pct = (
    cross_df
    .withColumn(
        "total",
        sf.col("0") + sf.col("1")
    )
    .withColumn(
        "Repaid_pct",
        sf.round(sf.col("0") / sf.col("total") * 100, 2)
    )
    .withColumn(
        "Defaulted_pct",
        sf.round(sf.col("1") / sf.col("total") * 100, 2)
    )
)

# Convert to pandas
heatmap_df = (
    cross_pct
    .select(
        "FLAG_OWN_CAR_TARGET",
        "Repaid_pct",
        "Defaulted_pct"
    )
    .toPandas()
    .set_index("FLAG_OWN_CAR_TARGET")
    .rename(
        index={
            "N": "No Car",
            "Y": "Owns Car"
        }
    )
)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".2f",
    cmap="Greens",
    linewidths=0.5,
    linecolor="black",
    cbar_kws={"label": "Percentage (%)"},
    ax=ax
)

ax.set_title(
    "Default Rate by Car Ownership",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Repayment Status")
ax.set_ylabel("Car Ownership")

plt.tight_layout()
plt.show()

#### Conclusion

##### 1. Asset Ownership as a Financial Cushion
* **Liquidity and Collateral Buffer:** Car ownership signals accumulated wealth or borrowing capacity. In moments of sudden income disruption (e.g., medical emergency, temporary job loss), an applicant who owns a vehicle possesses a tangible, liquid asset that can be sold, refinanced, or leveraged for secondary funds to stay current on debt obligations.
* **Solvency Threshold:** Acquiring and maintaining a vehicle requires a baseline level of disposable income that non-car owners may not consistently maintain.

##### 2. Income & Employment Trajectory
* **Commute & Job Mobility:** Reliable personal transportation often grants access to higher-paying, stable employment opportunities that are not constrained by public transit routes or operating schedules.
* **Higher Earnings Baseline:** Car owners in consumer credit datasets skew toward higher median incomes, which directly lowers the Debt-to-Income (DTI) ratio and increases installment coverage.

##### 3. Demographic Confounding Factors
* **Age & Career Stage:** Car ownership naturally correlates with age and career maturity. Older borrowers generally possess longer credit histories and established savings habits, both of which drive down default rates.
* **Family Status:** Borrowers with vehicles are more likely to head multi-person households with dual income streams, offering secondary income protection if one household member faces financial stress.

##### Why the Gap Is Relatively Small (1.26 Percentage Points)
* **Vehicle Operating Overhead:** Unlike real estate, vehicles are depreciating assets that require recurring expenditures (fuel, insurance, repairs). High maintenance costs on older vehicles can sometimes act as a net drain on monthly cash flow.
* **Unfiltered Car Quality:** The binary `FLAG_OWN_CAR` indicator lumps high-value, recent-model vehicles together with low-value, aging automobiles that offer limited financial buffer during prolonged crises.

---

#### `FLAG_OWN_REALTY` (Real Estate Ownership)
A binary flag (`Y` = Yes, `N` = No) indicating whether the applicant owns real estate (a house, apartment, or land).

In [0]:
app_train.groupBy("FLAG_OWN_REALTY").count().show()

In [0]:
# aggregate in Spark
realty_df = (
    app_train
    .groupBy("FLAG_OWN_REALTY")
    .count()
    .toPandas()
)

realty_df["FLAG_OWN_REALTY"] = realty_df["FLAG_OWN_REALTY"].map({
    "Y": "Owns Realty",
    "N": "No Realty"
})

# Plot pie chart
fig, ax = plt.subplots(figsize=(7, 5))

ax.pie(
    realty_df["count"],
    labels=realty_df["FLAG_OWN_REALTY"],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "edgecolor": "black"
    }
)

ax.set_title(
    "Distribution of Realty Ownership",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

#### Repayment Risk on Owning a Real Estate

In [0]:
# Crosstab
cross_df = app_train.crosstab("FLAG_OWN_REALTY", "TARGET")

cross_pct = (
    cross_df
    .withColumn(
        "total",
        sf.col("0") + sf.col("1")
    )
    .withColumn(
        "Repaid_pct",
        sf.round(sf.col("0") / sf.col("total") * 100, 2)
    )
    .withColumn(
        "Defaulted_pct",
        sf.round(sf.col("1") / sf.col("total") * 100, 2)
    )
)

# Convert to pandas
heatmap_df = (
    cross_pct
    .select(
        "FLAG_OWN_REALTY_TARGET",
        "Repaid_pct",
        "Defaulted_pct"
    )
    .toPandas()
    .set_index("FLAG_OWN_REALTY_TARGET")
    .rename(
        index={
            "N": "No Realty",
            "Y": "Owns Realty"
        }
    )
)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    linewidths=0.5,
    linecolor="black",
    cbar_kws={"label": "Percentage (%)"},
    ax=ax
)

ax.set_title(
    "Default Rate by Real Estate Ownership",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Repayment Status")
ax.set_ylabel("Real Estate Ownership")

plt.tight_layout()
plt.show()

#### Business & Domain Rationale: Why Real Estate Owners Have a Slightly Lower Default Rate?

##### 1. Appreciation & Stability vs. Depreciation
* **Appreciating Net Worth Cushion:** Unlike cars, which depreciate rapidly and require recurring maintenance, real estate generally retains or grows in value over time. Owning property builds long-term equity, offering a solid safety net during financial crises.
* **Lower Housing Expense Volatility:** Homeowners are protected from sudden rent hikes and lease terminations, leading to more predictable fixed living costs and steady cash flow to service loan payments.

##### 2. High Rootedness & Long-Term Financial Commitment
* **Risk Aversion and Credit History:** Securing property typically requires a proven track record of saving for down payments and maintaining a long-term credit history. This established financial discipline directly translates to more responsible debt repayment behavior.
* **Geographic & Employment Stability:** Property owners are less mobile and tend to have deeply rooted careers, making sudden income disruptions or unannounced moves significantly less frequent.

---

In [0]:
app_train.groupBy("CNT_CHILDREN").count().show()

In [0]:
children_df = (
    app_train
    .groupBy("CNT_CHILDREN")
    .count()
    .orderBy("CNT_CHILDREN")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=children_df,
    x="CNT_CHILDREN",
    y="count",
    ax=ax,
    palette="plasma",
    edgecolor="black",
    hue="CNT_CHILDREN"
)

# Add count labels
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f")

ax.set_title(
    "Distribution of Number of Children",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Number of Children")
ax.set_ylabel("Applicant Count")

plt.tight_layout()
plt.show()

#### Why 0 Children Count is Dominating?
The distribution plot shows that the vast majority of applicants (~70%) have **0 children**, with counts dropping off rapidly for 1, 2, or more children. Over 98.5% of all applicants in the dataset have 2 or fewer children, while counts above 3 are extremely rare, extending to extreme outliers like 14 or 19 children.

The reason behind this heavy concentration at zero is twofold. First, loan applications heavily feature young, early-career adults who have not yet started a family. Second, credit applications strictly define `CNT_CHILDREN` as **financially dependent minor children (under 18)**. This means older applicants whose children are grown up and independent naturally revert to 0 on their applications, combining two distinct life stages into the exact same category.

From a modeling perspective, because this zero-inflated feature is skewed by age, looking at child counts alone can be misleading. Younger applicants with 0 children carry a higher default risk due to shorter employment histories, whereas older "empty nesters" with 0 children carry much lower risk due to accumulated financial stability.

---

#### `AMT_INCOME_TOTAL` (Annual Income)
The total annual income reported by the applicant. It Measures baseline earning power and repayment capacity. On its own, raw income can be noisy or right-skewed, but it sets the ceiling for how much debt an applicant can safely service.

In [0]:
# summary stats
app_train.select("AMT_INCOME_TOTAL").describe().show()

In [0]:
quantiles = app_train.approxQuantile(
    "AMT_INCOME_TOTAL",
    [0.0, 0.25, 0.50, 0.75, 0.9, 0.95, 1.0],
    0.01 # relative error
)

print(f"Min: {quantiles[0]:,.2f}")
print(f"Q1: {quantiles[1]:,.2f}")
print(f"Median: {quantiles[2]:,.2f}")
print(f"Q3: {quantiles[3]:,.2f}")
print(f"90%ile: {quantiles[4]:,.2f}")
print(f"95%ile: {quantiles[5]:,.2f}")
print(f"Max: {quantiles[6]:,.2f}")

In [0]:
# 10 most highest income values
app_train.select("AMT_INCOME_TOTAL").orderBy("AMT_INCOME_TOTAL", ascending=False).limit(10).show()

In [0]:
# box plot
app_train.select("AMT_INCOME_TOTAL").plot.box()

----

#### `AMT_CREDIT` (Loan Credit Amount)
The total principal credit amount of the loan granted/requested. It Represents the total liability the applicant is taking on. Higher credit amounts increase absolute default risk unless balanced by high income or substantial asset collateral.

In [0]:
# summary stats
app_train.select("AMT_CREDIT").describe().show()

In [0]:
# box plot
app_train.select("AMT_CREDIT").plot.box()

In [0]:
# histogram plot
app_train.select("AMT_CREDIT").plot.hist(title="Histogram Plot of AMT_CREDIT")

In [0]:
quantiles = app_train.approxQuantile(
    "AMT_CREDIT",
    [0.0, 0.25, 0.50, 0.75, 0.9, 0.95, 1.0],
    0.01 # relative error
)

print(f"Min: {quantiles[0]:,.2f}")
print(f"Q1: {quantiles[1]:,.2f}")
print(f"Median: {quantiles[2]:,.2f}")
print(f"Q3: {quantiles[3]:,.2f}")
print(f"90%ile: {quantiles[4]:,.2f}")
print(f"95%ile: {quantiles[5]:,.2f}")
print(f"Max: {quantiles[6]:,.2f}")

---

#### `AMT_ANNUITY` (Loan Payment Annuity)
The periodic (monthly) loan repayment installment amount.

In [0]:
# summary stats
app_train.select("AMT_ANNUITY").summary().show()

In [0]:
app_train.select("AMT_ANNUITY").plot.hist(title="Histogram Plot of AMT_ANNUITY")

In [0]:
app_train.select(sf.log("AMT_ANNUITY")).plot.hist(title="Histogram Plot of AMT_ANNUITY (Log Transformed)")

*Note: The log transformation has transformed the right skewed distribution into approx normal distribution*

In [0]:
# box plot
app_train.select("AMT_ANNUITY").plot.box()

Looking at the summary statistics, the average monthly payment is around 27,108, with small micro-loans starting as low as 1,615. However, the distribution has a long right tail extending up to a maximum of 258,025.5, showing that the dataset spans both everyday consumer purchases and heavy, high-tier monthly obligations.

---

#### `NAME_TYPE_SUITE`(Who Accompanied the Applicant)

##### What It Is?
It records whether the applicant was **alone or accompanied by someone** (e.g., `Unaccompanied`, `Spouse, partner`, `Family`, `Children`, `Group of people`) when applying for the loan.

##### Why It Even Matters?
1. **Behavioral Risk Indicator:** Applying with a spouse or family member often signals **shared household financial responsibility** and higher social accountability, which statistically correlates with lower default rates compared to applying completely alone.
2. **Point-of-Sale Context:** It reflects *where and how* the loan was taken out—applicants accompanied by family are more likely in a physical store making planned household purchases (like appliances or furniture), whereas unaccompanied applicants may be seeking quick, discretionary cash financing.

In [0]:
app_train.groupBy("NAME_TYPE_SUITE").count().show()

Other_A and Other_B are generalized fallback categories used by loan officers at point-of-sale terminals or branch offices when an accompanying person did not fit standard options like Unaccompanied, Spouse, partner, Family, or Children.

##### What They Represent??
Other_A: Typically represents non-family acquaintances who have a close or structured relationship with the applicant—such as a fiancé(e), roommate, co-worker, or business partner.

Other_B: Typically represents casual or distant acquaintances—such as a friend, neighbor, distant relative, or acquaintance who merely walked into the branch or store with the applicant.

In [0]:
app_train.crosstab("CODE_GENDER", "NAME_TYPE_SUITE").show()

---

In [0]:
app_train.groupBy("NAME_INCOME_TYPE").count().show()

#### `NAME_INCOME_TYPE` (Applicant Income & Employment Source)

##### What It Is
It records the **primary source or classification of the applicant's income** when they submit their loan application (e.g., `Working`, `Commercial associate`, `Pensioner`, `State servant`, `Unemployed`, `Student`, `Businessman`, `Maternity leave`).

##### Why It Matters

**Direct Indicator of Financial Stability:** 
   Income category directly shapes an applicant's cash-flow predictability and vulnerability to economic shocks:
   * **`Working` / `Commercial associate`:** Form the majority of applicants; income depends heavily on industry performance and continuous employment.
   * **`State servant`:** Civil/government workers carry lower default risk due to high job security and guaranteed salaries.
   * **`Pensioner`:** Fixed, reliable government income, yielding low default rates despite lower average total income amounts.
   * **`Unemployed` / `Maternity leave` / `Student`:** Small sample sizes that carry substantially higher default risk due to interrupted or nonexistent cash flows.

In [0]:
income_df = (
    app_train
    .groupBy("NAME_INCOME_TYPE")
    .count()
    .orderBy(sf.desc("count"))
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=income_df,
    y="NAME_INCOME_TYPE",
    x="count",
    ax=ax,
    edgecolor="black",
    palette="viridis",
    hue="NAME_INCOME_TYPE",
)

# Add count labels
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)

ax.set_title(
    "Distribution of Income Types",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Applicant Count")
ax.set_ylabel("Income Type")

plt.tight_layout()
plt.show()

#### Repayment Risk on NAME_INCOME_TYPE

In [0]:
income_default_df = (
    app_train
    .groupBy("NAME_INCOME_TYPE")
    .agg(
        sf.count("*").alias("total"),
        sf.sum("TARGET").alias("defaults")
    )
    .withColumn(
        "default_pct",
        sf.round(
            sf.col("defaults") / sf.col("total") * 100,
            2
        )
    )
    .orderBy(sf.desc("default_pct"))
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=income_default_df,
    y="NAME_INCOME_TYPE",
    x="default_pct",
    ax=ax,
    edgecolor="black",
    palette="bright",
    hue="NAME_INCOME_TYPE",
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f%%",
        padding=3
    )

ax.set_title(
    "Default Rate by Income Type",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Default Rate (%)")
ax.set_ylabel("Income Type")

plt.tight_layout()
plt.show()

#### How Distribution Volume Explains the Default Rate Anomalies?

##### 1. The "Tiny Sample Size" (Extreme Default Rates Exposed)
After cross-referencing this count distribution with the default rates reveals why those extreme rates (40% for Maternity leave, 0% for Students/Businessmen) exist:

* **`Maternity leave` (40.0% Default Rate):** Represents **only 5 total applicants** in the entire dataset. A 40% rate simply means **2 out of 5** defaulted. A single extra default swings the rate by a massive 20 percentage points.
* **`Unemployed` (36.4% Default Rate):** Based on **only 22 applicants** (8 defaulted).
* **`Student` & `Businessman` (0.0% Default Rate):** Based on **18** and **10 applicants** respectively. Zero defaults in a tiny sample of 10–18 people is pure statistical chance, not proof that businessmen or students never default.

##### 2. The Core 99.9% Population (The Reliable Signal)
Over **99.9% of all loan applications** (~307,456 out of ~307,511) belong to just the top four categories:

1. **`Working` (158,774 applicants | ~51.6% of data):** The baseline workforce. With 158k+ samples, its **9.59% default rate** is rock-solid statistical ground truth.
2. **`Commercial associate` (71,617 applicants | ~23.3% of data):** Corporate/business employees showing a consistent **7.48% default rate**.
3. **`Pensioner` (55,362 applicants | ~18.0% of data):** A massive group proving that state-backed stability yields a reliable **5.39% default rate**.
4. **`State servant` (21,703 applicants | ~7.1% of data):** High job security translating to a low **5.75% default rate**.

---

#### NAME_EDUCATION_TYPE (Education of the Applicant)

In [0]:
app_train.groupBy("NAME_EDUCATION_TYPE").count().show(truncate=-1)

In [0]:
education_df = (
    app_train
    .groupBy("NAME_EDUCATION_TYPE")
    .count()
    .toPandas()
)

fig, ax = plt.subplots(figsize=(8, 8))

wedges, texts, autotexts = ax.pie(
    education_df["count"],
    labels=None,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "edgecolor": "black",
        "linewidth": 1
    }
)

ax.legend(
    wedges,
    education_df["NAME_EDUCATION_TYPE"],
    title="Education Type",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

ax.set_title("Distribution of Education Types")

plt.tight_layout()
plt.show()

#### `NAME_EDUCATION_TYPE` Overview & Domain Context

##### Cultural Context: Why the Naming Sounds Unfamiliar
These category names reflect the educational system common in **Eastern Europe and the post-Soviet region** (where Home Credit operates), which maps closely to the **ISCED** (International Standard Classification of Education) framework:

##### Category Definitions & Market Breakdown

1. **`Secondary / secondary special` (218,391 applicants | ~71.0%)**
   * **Equivalent:** High School Diploma OR Vocational / Technical College (*Technikum*).
   * **Description:** Represents completed secondary education. "Secondary special" specifically means trade-focused education (e.g., certified technicians, nurses, electricians, mechanics, bookkeepers).
   * **Credit Signal:** The core baseline of the applicant pool. Holds stable blue-collar or technical administrative jobs.

2. **`Higher education` (74,863 applicants | ~24.3%)**
   * **Equivalent:** University Graduate (Bachelor’s or Master’s degree).
   * **Description:** Full completion of an accredited university program.
   * **Credit Signal:** Strongest predictor of financial resilience. University graduates consistently command higher baseline salaries, lower unemployment rates during crises, and the lowest default risk in retail banking.

3. **`Incomplete higher` (10,277 applicants | ~3.3%)**
   * **Equivalent:** Some University / Dropped Out / Currently Enrolled Student.
   * **Description:** The applicant attended university for at least a few semesters but did not complete the degree program, or is a young student currently in university.
   * **Credit Signal:** Mixed risk profile. Young current students often carry higher risk due to low liquid savings, whereas working adults with incomplete degrees fall between general high school and university graduate risk profiles.

4. **`Lower secondary` (3,816 applicants | ~1.2%)**
   * **Equivalent:** Middle School / Basic Compulsory Schooling (typically up to ~9th grade).
   * **Description:** Completed basic primary/middle school education but did not complete high school or vocational school.
   * **Credit Signal:** Highest default risk group. Correlates heavily with low-skilled manual labor, irregular/seasonal employment, and lower earning potential.

5. **`Academic degree` (164 applicants | ~0.05%)**
   * **Equivalent:** Doctorate / Ph.D. / Post-Doctoral / Research Professor.
   * **Description:** Post-graduate research degrees above a standard Master's degree.
   * **Credit Signal:** Extremely rare in general retail micro-lending. Due to the tiny sample size ($N=164$), treating this as a distinct category in tree models can lead to overfitting; it is often best grouped into `Higher education`.

*Note: This is an ordinal feature*

---

#### NAME_FAMILY_STATUS (Maritial Status)

In [0]:
family_df = (
    app_train
    .groupBy("NAME_FAMILY_STATUS")
    .count()
    .orderBy(sf.desc("count"))
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=family_df,
    y="NAME_FAMILY_STATUS",
    x="count",
    ax=ax,
    edgecolor="black",
    palette="Greens_r",
    hue="NAME_FAMILY_STATUS",
)

# Add count labels
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)

ax.set_title(
    "Distribution of Family Status",
    fontsize=14,
    fontweight="bold",
    pad=15,
)

ax.set_xlabel("Applicant Count")
ax.set_ylabel("Family Status")

plt.tight_layout()
plt.show()

##### What is Civil Marriage?
**Civil marriage** refers to cohabitation or a common-law partnership where a couple lives together long-term without an official legal marriage certificate. While in some Western countries "civil marriage" means a government-registered non-religious union, on Eastern European credit forms, it is colloquially used to distinguish couples sharing a household from those with formal marriage licenses.

For credit risk, this distinction matters because cohabitating partners share daily household expenses, but lack the legal joint liability for debt that binds legally married couples. Consequently, applicants in a civil marriage sit in a moderate risk tier—statistically showing slightly higher default rates than legally married borrowers (who have the lowest risk), but lower default rates than single or divorced applicants.

#### Repayment Risk on Maritial Status

In [0]:
# Calculate default rate by family status
family_risk_df = (
    app_train
    .groupBy("NAME_FAMILY_STATUS")
    .agg(
        sf.count("*").alias("total_applicants"),
        sf.sum("TARGET").alias("defaults")
    )
    .withColumn(
        "default_rate_pct",
        sf.round(
            sf.col("defaults") / sf.col("total_applicants") * 100,
            2
        )
    )
    .orderBy(sf.desc("default_rate_pct"))
    .toPandas()
)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=family_risk_df,
    y="NAME_FAMILY_STATUS",
    x="default_rate_pct",
    ax=ax,
    edgecolor="black",
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f%%",
        padding=3
    )

ax.set_title(
    "Default Rate by Family Status",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Default Rate (%)")
ax.set_ylabel("Family Status")

plt.tight_layout()
plt.show()

#### Iterpretation
The high default rates for civil marriage (9.94%) and single applicants (9.81%) stem from single-income vulnerability and age distribution. Single borrowers skew younger with shorter employment histories and less savings. While cohabitating "civil marriage" partners share household costs, they lack joint legal debt liability, meaning an individual financial shock isn't legally cushioned by the partner's income.

Legally married applicants (7.56%) benefit from dual-income buffers, shared financial goals, and joint asset accumulation, making them far more resilient to temporary job loss or expense spikes. Separated applicants (8.19%) face a moderate risk increase driven by the immediate financial disruption, legal expenses, and asset splits that come with household separation.

The lowest default rate among widows (5.82%) is a demographic masking effect driven primarily by applicant age. Widows in credit datasets are overwhelmingly older individuals and pensioners. As we saw with income types, pensioners receive guaranteed, predictable state income and generally have lower debt-to-income overhead, making age the underlying driver behind their low risk profile.

---

#### NAME_HOUSING_TYPE

In [0]:
app_train.groupBy("NAME_HOUSING_TYPE").count().show()

In [0]:
housing_count_df = (
    app_train
    .groupBy("NAME_HOUSING_TYPE")
    .count()
    .orderBy("count")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 5))

# Stems
ax.hlines(
    y=housing_count_df["NAME_HOUSING_TYPE"],
    xmin=0,
    xmax=housing_count_df["count"],
    color="orange",
    linewidth=2
)

# Dots
ax.plot(
    housing_count_df["count"],
    housing_count_df["NAME_HOUSING_TYPE"],
    "o",
    markersize=8
)

# Count labels
for _, row in housing_count_df.iterrows():
    ax.text(
        row["count"] + 500,
        row["NAME_HOUSING_TYPE"],
        f"{row['count']:,}",
        va="center"
    )

ax.set_xlabel("Applicant Count")
ax.set_ylabel("Housing Type")
ax.set_title(
    "Distribution of Housing Types",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

##### Default Rate per Housing Type

In [0]:
housing_default_pd = (
    app_train
    .groupBy("NAME_HOUSING_TYPE")
    .agg(
        sf.round(sf.mean("TARGET") * 100, 2).alias("default_rate_pct")
    )
    .orderBy(sf.desc("default_rate_pct"))
    .select("NAME_HOUSING_TYPE", "default_rate_pct")
    .toPandas()
)

housing_default_pd

In [0]:
fig, ax = plt.subplots(figsize=(8, 4))

sns.barplot(
    data=housing_default_pd,
    x="NAME_HOUSING_TYPE",
    y="default_rate_pct",
    palette="Reds",
    edgecolor="black",
    linewidth=1.2,
    ax=ax,
    hue="NAME_HOUSING_TYPE",
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f%%",
        padding=3,
        fontsize=10,
        fontweight="bold"
    )

ax.set_title(
    "Default Rate by Housing Type",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Housing Type")
ax.set_ylabel("Default Rate (%)")

plt.xticks(rotation=20)
sns.despine()
plt.tight_layout()
plt.show()

---

#### `REGION_POPULATION_RELATIVE`

This column represents the normalized population density of the region where the client lives relative to the rest of the country. A higher decimal value indicates that the applicant resides in a heavily populated urban center or major city, while a smaller value points to a rural or sparsely populated area.

In [0]:
app_train.select("REGION_POPULATION_RELATIVE").summary().show()

In [0]:
app_train.select("REGION_POPULATION_RELATIVE").plot.hist(title="Histogram Plot of REGION_POPULATION_RELATIVE")

##### 📈 Key Insights from `REGION_POPULATION_RELATIVE`

##### 1. Core Mass-Market Base
The vast majority of loan applicants live in regions with relative population densities between **0.007** and **0.036**, peaking around **0.015–0.022**. This shows that Home Credit’s primary customer base resides in standard mid-sized cities, towns, and suburban areas rather than remote rural zones or mega-cities.

##### 2. Discrete Region Encoding
The wide empty gaps around **0.04** and **0.06** show that this isn't a smooth, continuous measurement. Instead, it acts as a masked proxy for specific administrative regions. Each bar represents a distinct group of cities or territories sharing the exact same population index.

##### 3. Metropolitan Outlier Clusters
The separate, smaller bars at **0.048** and **0.072** highlight applicants living in high-density metropolitan hubs and capital cities. Borrowers in these high-density regions often face higher costs of living and higher income tiers, giving them a distinct risk profile compared to the main suburban cluster.

##### Relationship Between AMT_INCOME_TOTAL and REGION_POPULATION_RELATIVE

In [0]:
app_train.select("AMT_INCOME_TOTAL", "REGION_POPULATION_RELATIVE").plot.scatter("AMT_INCOME_TOTAL", "REGION_POPULATION_RELATIVE")

---

#### 🎂 DAYS_BIRTH

`DAYS_BIRTH` measures the applicant's age at the time of application.

In [0]:
app_train.select("DAYS_BIRTH").summary().show()

---

#### DAYS_EMPLOYED

`DAYS_EMPLOYED` measures how many days before the application date the customer started working at their current job. 

In the raw dataset, these values are stored as **negative numbers**. For example, a value of `-1000` means the applicant has been employed at their current job for 1,000 days (roughly 2.7 years).

#### How to Interpret It
* **Longer Tenure (Large Negative Values):** A number like `-3000` (about 8 years on the job) signals strong employment stability, steady income, and generally lower credit risk.
* **Shorter Tenure (Values Close to 0):** A number like `-60` means the applicant joined their employer just two months ago, indicating higher job instability and higher potential default risk.

In [0]:
app_train.select("DAYS_EMPLOYED").summary().show()

In [0]:
# values above 0
app_train.filter(sf.col("DAYS_EMPLOYED")>0).count()

In [0]:
# only equals to '365243' 
app_train.filter(sf.col("DAYS_EMPLOYED")==365243).count()

##### The Meaning of `365243`

In the Home Credit dataset, **`365243`** is an artificial placeholder code used by the bank to represent **pensioners, retirees, or unemployed applicants**. 

If we divide `365243` by 365 days, it equals roughly **1,000 years**. Because a customer cannot be employed for 1,000 years into the future, legacy banking systems inserted this arbitrary constant whenever an applicant didn't have an active employer, avoiding empty or `NULL` values in their database.

In [0]:
app_train.filter(sf.col("DAYS_EMPLOYED")==0).show()

**`DAYS_EMPLOYED == 0`** means the applicant started working at their current job on the **exact same day** they submitted their loan application.

In [0]:
# box plot after filtering positives
app_train.filter(sf.col("DAYS_EMPLOYED")<0).select("DAYS_EMPLOYED").plot.box()

In [0]:
app_train.show(5)

### 📈 Data Drift Report and Tests

In [0]:
app_train.printSchema()

In [0]:
# categorical features
cat_cols = [
    col_name
    for col_name, dtype in app_train.drop("SK_ID_CURR", "TARGET").dtypes
    if dtype in ["string"]
]

# numerical features
num_cols = [
    col_name
    for col_name, dtype in app_train.drop("SK_ID_CURR", "TARGET").dtypes
    if dtype not in ["string"]
]

# primary key
id_col = "SK_ID_CURR"

In [0]:
# converting spark dataframes -> pandas dataframes because evidently dont support pyspark dfs yet
train_pd = app_train.drop("TARGET").toPandas()
test_pd = app_test.toPandas()

In [0]:
# defining the train and test set schema
data_definition = DataDefinition(
    numerical_columns=num_cols,
    categorical_columns=cat_cols,
    id_column=id_col,
)

reference_data = Dataset.from_pandas(
    train_pd,
    data_definition=data_definition
)

current_data = Dataset.from_pandas(
    test_pd,
    data_definition=data_definition
)

In [0]:
# creating the report
report = Report(
    metrics=[
        DataDriftPreset()
    ],
    include_tests=True, # tests
)

my_eval = report.run(
    reference_data=reference_data,
    current_data=current_data
)

In [0]:
# saving report file in current working dir
# my_eval.save_html("01b_data_drift_report.html")